<a href="https://colab.research.google.com/github/Varsanrecp/ML-portfolio/blob/main/RAG_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [14]:
! pip GoogleGenerativeAI

In [35]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'

if not os.getenv("LANGCHAIN_API_KEY"):
    os.environ['LANGCHAIN_API_KEY'] = getpass.getpass("Enter your LangChain API key: ")
os.environ['LANGCHAIN_API_KEY'] = os.environ['LANGCHAIN_API_KEY']
print("API key loaded into environment variables (not printed).")

API key loaded into environment variables (not printed).


In [36]:
import os, getpass

if not os.getenv("GOOGLE_API_KEY"):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter your Google / Gemini API key (from Google AI Studio): ")

# set GEMINI_API_KEY too for compatibility with some examples
os.environ['GOOGLE_API_KEY'] = os.environ['GOOGLE_API_KEY']

print("API key loaded into environment variables (not printed).")

API key loaded into environment variables (not printed).


In [12]:
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import GoogleGenerativeAIEmbeddings, GoogleGenerativeAI
import os

#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed
# Ensure the API key is passed to GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=os.environ["GOOGLE_API_KEY"])
vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=embeddings)

retriever = vectorstore.as_retriever()

#### RETRIEVAL and GENERATION ####

# Prompt
prompt = hub.pull("rlm/rag-prompt")

# LLM
# Ensure the API key is passed to GoogleGenerativeAI
llm = GoogleGenerativeAI(model="gemini-2.5-pro", temperature=0.0, google_api_key=os.environ["GOOGLE_API_KEY"])

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Question
rag_chain.invoke("What is Task Decomposition?")

'Task decomposition is the process of breaking down a complex task into smaller, simpler, and more manageable steps or subgoals. This planning technique helps enhance performance on difficult tasks by transforming a large problem into multiple manageable ones. It can be accomplished through methods like Chain of Thought prompting, using task-specific instructions, or with human input.'

In [13]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

In [14]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question, "cl100k_base")

8

In [15]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings, GoogleGenerativeAI

embd = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
len(query_result)

3072

In [16]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.9015230867829924


In [17]:
#### INDEXING ####

# Load blog
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

In [20]:
# Split
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)


In [21]:
# Index
from langchain_google_genai import GoogleGenerativeAIEmbeddings, GoogleGenerativeAI
from langchain_community.vectorstores import Chroma


vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=embeddings)

retriever = vectorstore.as_retriever()

In [24]:
# Index
from langchain_google_genai import GoogleGenerativeAIEmbeddings, GoogleGenerativeAI
from langchain_community.vectorstores import Chroma


retriever = vectorstore.as_retriever(search_kwargs={"k": 1})


In [25]:
docs = retriever.get_relevant_documents("What is Task Decomposition?")

/tmp/ipython-input-4059233835.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents("What is Task Decomposition?")


In [26]:

len(docs)

1

In [27]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, GoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [28]:
llm = GoogleGenerativeAI(model="gemini-2.5-pro", temperature=0.0, google_api_key=os.environ["GOOGLE_API_KEY"])

In [29]:

# Chain
chain = prompt | llm

In [30]:
chain.invoke({"context":docs,"question":"What is Task Decomposition?"})

'Based on the provided context, task decomposition can be done in one of three ways:\n\n1.  By an LLM with simple prompting, such as "Steps for XYZ." or "What are the subgoals for achieving XYZ?".\n2.  By using task-specific instructions, for example, "Write a story outline." for writing a novel.\n3.  With human inputs.'

In [31]:
from langchain import hub
prompt_hub_rag = hub.pull("rlm/rag-prompt")

In [32]:
prompt_hub_rag

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [33]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")

'Based on the provided context, task decomposition can be done in one of three ways:\n\n1.  By an LLM with simple prompting, such as "Steps for XYZ." or "What are the subgoals for achieving XYZ?".\n2.  By using task-specific instructions, for example, "Write a story outline." for writing a novel.\n3.  With human inputs.'